In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

In [3]:
dataset_path = r"C:\Users\user5\OneDrive\Desktop\Deep-Learning\Dataset\101_ObjectCategories"

In [4]:
categories = ["airplanes","butterfly","elephant","Motorbikes","sunflower"]

In [6]:
images = []
labels = []

for index,category in enumerate(categories):
    folder = os.path.join(dataset_path,category)
    for file in os.listdir(folder):
        file_path = os.path.join(folder,file)
        image = load_img(file_path, target_size=(224,224))
        image = img_to_array(image)
        labels.append(index)
        images.append(image)
X = np.array(images)
Y = np.array(labels)

In [7]:
X = preprocess_input(X)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,Y,test_size=0.2,random_state=42,stratify=Y
)

print("Training images: ",X_train.shape)
print("Test images: ",y_train.shape)

Training images:  (1470, 224, 224, 3)
Test images:  (1470,)


In [11]:
vgg16 = VGG16(weights = "imagenet", include_top=False, input_shape=(224,224,3))

for layer in vgg16.layers:
    if layer.name.startswith("block5"):
        layer.trainable = True
    else:
        layer.trainable = False

In [12]:
for layer in vgg16.layers:
    print(layer.name, "->", layer.trainable)

input_layer_1 -> False
block1_conv1 -> False
block1_conv2 -> False
block1_pool -> False
block2_conv1 -> False
block2_conv2 -> False
block2_pool -> False
block3_conv1 -> False
block3_conv2 -> False
block3_conv3 -> False
block3_pool -> False
block4_conv1 -> False
block4_conv2 -> False
block4_conv3 -> False
block4_pool -> False
block5_conv1 -> True
block5_conv2 -> True
block5_conv3 -> True
block5_pool -> True


In [15]:
y_train_cat = to_categorical(y_train, num_classes = len(categories))
y_test_cat = to_categorical(y_test, num_classes=len(categories))

In [16]:
model = Sequential([
    Input(shape=(224, 224, 3)),
    vgg16,
    GlobalAveragePooling2D(),
    Dense(256, activation="relu"),
    Dropout(0.5),
    Dense(128, activation="relu"),
    Dense(len(categories), activation="softmax")
])

In [17]:
model.summary()

# Cell 8: Compile   (f. Adam optimizer + categorical cross-entropy)
# a small learning rate is used because block 5 is being fine-tuned
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)                   │ (None, 7, 7, 512)           │      14,714,688 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 512)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 5)                   │             645 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 14,879,557 (56.76 MB)

 Trainable params: 7,244,293 (27.63 MB)

 Non-trainable params: 7,635,264 (29.13 MB)

In [19]:
history = model.fit(
    X_train, y_train_cat,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/10
 6/37 ━━━━━━━━━━━━━━━━━━━━ 2:25 5s/step - accuracy: 0.6510 - loss: 1.3727

KeyboardInterrupt: 

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, Y_test_cat)
print("Test Accuracy:", test_accuracy)

# Cell 11: Training and validation accuracy   (h. Display accuracy)
print("\nEpoch-wise accuracy:")
for i in range(len(history.history["accuracy"])):
    print("Epoch", i + 1,
          "| Train Acc:", round(history.history["accuracy"][i], 4),
          "| Val Acc:", round(history.history["val_accuracy"][i], 4))

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.show()

# Cell 12: Predict categories for sample test images
predictions = model.predict(X_test[:10])
predicted_classes = np.argmax(predictions, axis=1)

for i in range(10):
    print("Actual:", categories[Y_test[i]],
          "| Predicted:", categories[predicted_classes[i]],
          "| Confidence:", round(float(np.max(predictions[i])), 3))